# Parallel Computing with JAX for Earth Scientists — SOLUTIONS

This is the solutions notebook accompanying `JAX_Parallel_Computing_for_Earth_Scientists.ipynb`.
It contains the same worked examples plus completed versions of every exercise.

Run the setup cell first, then work through in order — later exercises reuse functions
defined in earlier sections.


In [ ]:
#@title Setup: install and import JAX
!pip install -q --upgrade jax jax[cuda12] 2>/dev/null || pip install -q --upgrade jax

import jax
import jax.numpy as jnp
import numpy as np
import time
import matplotlib.pyplot as plt

print("JAX version:", jax.__version__)
print("Devices JAX can see:", jax.devices())


---
## Section 1 — Warm-up: what does `jit` actually buy you?


In [ ]:
def spectral_correction(x):
    """A stand-in for a per-pixel spectral correction (e.g. NDVI-style),
    repeated a few times to give jit something to optimize."""
    for _ in range(10):
        x = jnp.sin(x) * jnp.cos(x) + jnp.exp(-x**2)
    return x

fast_correction = jax.jit(spectral_correction)

# A "satellite image": 1000 x 1000 grid of pixel reflectance values
x = jnp.ones((1000, 1000))

# Warm-up call: this is where tracing + compilation happens (slow, one-time cost)
fast_correction(x).block_until_ready()

# Time the non-jitted version
start = time.time()
for _ in range(20):
    spectral_correction(x).block_until_ready()
no_jit_time = time.time() - start

# Time the jitted version
start = time.time()
for _ in range(20):
    fast_correction(x).block_until_ready()
jit_time = time.time() - start

print(f"Without jit: {no_jit_time:.4f}s")
print(f"With jit:    {jit_time:.4f}s")
print(f"Speedup:     {no_jit_time / jit_time:.1f}x")


### Exercise 1 — Solution

We repeat the same comparison at a larger grid size. The speedup from `jit` typically
**grows** with array size (up to a point): with a bigger array, each individual op has more
work to do relative to its fixed Python/dispatch overhead in the non-jitted case, but the
*compiled* version also has more genuine parallel work to spread across the accelerator's
cores/lanes, and fusing 10 chained elementwise ops into one kernel avoids re-reading/writing
a much larger array to memory 10 times. Eventually, for extremely large arrays, both
versions become compute/memory-bound and the *relative* speedup can shrink again — but for
this range (1000x1000 to 5000x5000) you should see the jitted version pull further ahead.


In [ ]:
# Exercise 1 solution
x_big = jnp.ones((5000, 5000))

# Warm-up
fast_correction(x_big).block_until_ready()

start = time.time()
for _ in range(20):
    spectral_correction(x_big).block_until_ready()
no_jit_time_big = time.time() - start

start = time.time()
for _ in range(20):
    fast_correction(x_big).block_until_ready()
jit_time_big = time.time() - start

print(f"(5000x5000) Without jit: {no_jit_time_big:.4f}s")
print(f"(5000x5000) With jit:    {jit_time_big:.4f}s")
print(f"(5000x5000) Speedup:     {no_jit_time_big / jit_time_big:.1f}x")
print(f"\nFor comparison, (1000x1000) speedup was: {no_jit_time / jit_time:.1f}x")


---
## Section 2 — Ensemble parameter sweeps with `vmap`


In [ ]:
def thaw_depth_model(k, days=jnp.arange(1, 121, dtype=jnp.float32)):
    """Estimate thaw depth (m) over `days` days of a melt season for
    thermal parameter k, using a simple Stefan-type approximation."""
    cumulative_degree_days = jnp.cumsum(jnp.ones_like(days))
    return k * jnp.sqrt(cumulative_degree_days) / 100.0

k_values = jnp.linspace(1.0, 10.0, 2000)

start = time.time()
results_loop = jnp.stack([thaw_depth_model(k) for k in k_values])
results_loop.block_until_ready()
loop_time = time.time() - start
print(f"Python loop over {len(k_values)} ensemble members: {loop_time:.4f}s")

batched_thaw_model = jax.jit(jax.vmap(thaw_depth_model, in_axes=(0, None)))
days = jnp.arange(1, 121, dtype=jnp.float32)
batched_thaw_model(k_values, days).block_until_ready()

start = time.time()
results_vmap = batched_thaw_model(k_values, days)
results_vmap.block_until_ready()
vmap_time = time.time() - start
print(f"vmap + jit over {len(k_values)} ensemble members: {vmap_time:.4f}s")
print(f"Speedup: {loop_time / vmap_time:.1f}x")
print("Results match:", jnp.allclose(results_loop, results_vmap, atol=1e-4))


### Exercise 2 — Solution

We add a melt-season start offset `t0` and sweep over a 2D grid of `(k, t0)` combinations
using nested `vmap`. The inner `vmap` batches over `t0` (holding `k` and `days` fixed), and
the outer `vmap` then batches that whole function over `k`.


In [ ]:
# Exercise 2 solution
def thaw_depth_model_v2(k, t0, days):
    """thaw_depth(t) = k * sqrt(max(cumulative_degree_days(t) - t0, 0))"""
    cumulative_degree_days = jnp.cumsum(jnp.ones_like(days))
    shifted = jnp.maximum(cumulative_degree_days - t0, 0.0)
    return k * jnp.sqrt(shifted) / 100.0

k_values_2 = jnp.linspace(1.0, 10.0, 50)
t0_values = jnp.linspace(0.0, 60.0, 50)
days = jnp.arange(1, 121, dtype=jnp.float32)

# Inner vmap: batch over t0, holding k and days fixed
over_t0 = jax.vmap(thaw_depth_model_v2, in_axes=(None, 0, None))

# Outer vmap: batch that whole function over k
over_k_and_t0 = jax.jit(jax.vmap(over_t0, in_axes=(0, None, None)))

results_2d = over_k_and_t0(k_values_2, t0_values, days)
print("Output shape (k, t0, days):", results_2d.shape)
assert results_2d.shape == (len(k_values_2), len(t0_values), len(days))

# Quick visualization: final thaw depth as a function of (k, t0)
plt.figure(figsize=(6, 5))
plt.imshow(
    results_2d[:, :, -1],
    origin="lower",
    extent=[t0_values.min(), t0_values.max(), k_values_2.min(), k_values_2.max()],
    aspect="auto",
    cmap="viridis",
)
plt.colorbar(label="Final thaw depth (m)")
plt.xlabel("t0 (melt season start offset, days)")
plt.ylabel("k (thermal parameter)")
plt.title("Final thaw depth across (k, t0) ensemble")
plt.show()


---
## Section 3 — A 2D diffusion model: heat/pollutant spreading in a river


In [ ]:
def laplacian(C):
    """5-point stencil Laplacian with zero-flux (reflective) boundaries."""
    C_up    = jnp.pad(C[:-1, :], ((1, 0), (0, 0)), mode="edge")
    C_down  = jnp.pad(C[1:, :],  ((0, 1), (0, 0)), mode="edge")
    C_left  = jnp.pad(C[:, :-1], ((0, 0), (1, 0)), mode="edge")
    C_right = jnp.pad(C[:, 1:],  ((0, 0), (0, 1)), mode="edge")
    return C_up + C_down + C_left + C_right - 4 * C

def diffusion_step(C, _, D=0.2, dt=1.0):
    return C + D * dt * laplacian(C), None

def run_diffusion(C0, n_steps, D=0.2, dt=1.0):
    step_fn = lambda C, _: diffusion_step(C, _, D=D, dt=dt)
    C_final, _ = jax.lax.scan(step_fn, C0, xs=None, length=n_steps)
    return C_final

run_diffusion_jit = jax.jit(run_diffusion, static_argnames=("n_steps",))

grid_size = 100
C0 = jnp.zeros((grid_size, grid_size))
C0 = C0.at[grid_size // 2, grid_size // 4].set(1000.0)
n_steps = 2000

run_diffusion_jit(C0, n_steps).block_until_ready()

start = time.time()
C_final = run_diffusion_jit(C0, n_steps)
C_final.block_until_ready()
scan_time = time.time() - start
print(f"lax.scan + jit, {n_steps} steps: {scan_time:.4f}s")

def run_diffusion_pyloop(C0, n_steps, D=0.2, dt=1.0):
    C = C0
    for _ in range(n_steps):
        C = C + D * dt * laplacian(C)
    return C

start = time.time()
C_final_loop = run_diffusion_pyloop(C0, n_steps)
C_final_loop.block_until_ready()
loop_time = time.time() - start
print(f"Python for-loop, {n_steps} steps: {loop_time:.4f}s")
print(f"Speedup: {loop_time / scan_time:.1f}x")


### Exercise 3 — Solution

We add a constant source term at the spill cell every step (a continuous discharge, rather
than a single instantaneous spill), and have `lax.scan` collect the total grid mass
(`C.sum()`) at every step via its `ys` output so we can plot it over time.


In [ ]:
# Exercise 3 solution
source_i, source_j = grid_size // 2, grid_size // 4
source_strength = 50.0

def diffusion_step_with_source(C, _, D=0.2, dt=1.0):
    C_new = C + D * dt * laplacian(C)
    C_new = C_new.at[source_i, source_j].add(source_strength * dt)
    total_mass = C_new.sum()
    return C_new, total_mass

def run_diffusion_with_source(C0, n_steps, D=0.2, dt=1.0):
    step_fn = lambda C, _: diffusion_step_with_source(C, _, D=D, dt=dt)
    C_final, mass_history = jax.lax.scan(step_fn, C0, xs=None, length=n_steps)
    return C_final, mass_history

run_source_jit = jax.jit(run_diffusion_with_source, static_argnames=("n_steps",))

C0_src = jnp.zeros((grid_size, grid_size))
n_steps_src = 3000
C_final_src, mass_history = run_source_jit(C0_src, n_steps_src)

plt.figure(figsize=(6, 4))
plt.plot(mass_history)
plt.xlabel("Time step")
plt.ylabel("Total mass in grid")
plt.title("Total pollutant mass over time with continuous source")
plt.show()

print("Mass is still rising at the end:", mass_history[-1] > mass_history[-100])
print("(With reflective/no-flux boundaries and a constant source, mass keeps growing —")
print(" it never reaches a true steady state unless we also add an outflow/sink term.)")


---
## Section 4 — Monte Carlo particle transport (embarrassingly parallel)


In [ ]:
def velocity_field(pos):
    x, y = pos
    u = 1.0 - (y - 0.5) ** 2 * 4.0
    v = 0.0
    return jnp.array([u, v])

def advect_particle(key, pos0, n_steps=200, dt=0.05, turbulence=0.05):
    def step(carry, _):
        pos, key = carry
        key, subkey = jax.random.split(key)
        vel = velocity_field(pos)
        noise = jax.random.normal(subkey, shape=(2,)) * turbulence
        pos_new = pos + dt * vel + jnp.sqrt(dt) * noise
        pos_new = pos_new.at[1].set(jnp.clip(pos_new[1], 0.0, 1.0))
        return (pos_new, key), pos_new

    (pos_final, _), trajectory = jax.lax.scan(step, (pos0, key), xs=None, length=n_steps)
    return trajectory

advect_all = jax.jit(
    jax.vmap(advect_particle, in_axes=(0, 0, None, None, None)),
    static_argnames=("n_steps",),
)


### Exercise 4 — Solution

We time a Python-loop version against the `vmap` version at two ensemble sizes. You should
see the loop version's time scale roughly linearly with `n_particles`, while the `vmap`
version's time barely changes between 500 and 5000 particles (up to a point limited by
device memory/core count) — so the *speedup factor* grows substantially with ensemble size.
This illustrates the general rule: vectorization overhead (tracing, compilation) is roughly
fixed, so it pays off more and more as the batch dimension grows; for very small batches
the fixed overhead can even make `vmap` a wash or slightly worse than a plain loop.


In [ ]:
# Exercise 4 solution
def run_loop_version(n_particles, n_steps=200, dt=0.05, turbulence=0.05):
    key = jax.random.PRNGKey(0)
    keys = jax.random.split(key, n_particles)
    y0 = jnp.linspace(0.05, 0.95, n_particles)
    pos0 = jnp.stack([jnp.zeros(n_particles), y0], axis=1)

    start = time.time()
    trajs = [advect_particle(keys[i], pos0[i], n_steps, dt, turbulence) for i in range(n_particles)]
    trajs = jnp.stack(trajs)
    trajs.block_until_ready()
    return time.time() - start

def run_vmap_version(n_particles, n_steps=200, dt=0.05, turbulence=0.05):
    key = jax.random.PRNGKey(0)
    keys = jax.random.split(key, n_particles)
    y0 = jnp.linspace(0.05, 0.95, n_particles)
    pos0 = jnp.stack([jnp.zeros(n_particles), y0], axis=1)

    # warm-up (compile)
    advect_all(keys, pos0, n_steps, dt, turbulence).block_until_ready()

    start = time.time()
    trajs = advect_all(keys, pos0, n_steps, dt, turbulence)
    trajs.block_until_ready()
    return time.time() - start

for n in [500, 5000]:
    loop_t = run_loop_version(n)
    vmap_t = run_vmap_version(n)
    print(f"n_particles={n:5d}  loop={loop_t:.4f}s  vmap={vmap_t:.4f}s  speedup={loop_t/vmap_t:.1f}x")

print("\nObservation: the loop time grows roughly in proportion to n_particles (each particle")
print("is compiled/dispatched independently), while the vmap time is dominated by a mostly")
print("fixed compilation cost plus genuinely parallel execution — so the speedup factor")
print("grows substantially as the ensemble gets bigger. For a handful of particles, vmap's")
print("fixed overhead may not be worth it; for thousands, it clearly is.")


---
## Section 5 (optional) — Bonus: `jax.grad` for inverse problems

(No exercise in this section in the main worksheet — included here for completeness /
reference, matching the main notebook.)


In [ ]:
D_true = 0.35
C0_obs = jnp.zeros((grid_size, grid_size)).at[grid_size // 2, grid_size // 4].set(1000.0)
C_observed = run_diffusion(C0_obs, n_steps=500, D=D_true)
key = jax.random.PRNGKey(1)
C_observed = C_observed + jax.random.normal(key, C_observed.shape) * 2.0

def loss_fn(D, C0, C_target, n_steps=500):
    C_pred = run_diffusion(C0, n_steps=n_steps, D=D)
    return jnp.mean((C_pred - C_target) ** 2)

grad_loss = jax.jit(jax.grad(loss_fn), static_argnames=("n_steps",))
loss_jit = jax.jit(loss_fn, static_argnames=("n_steps",))

D_estimate = 0.1
learning_rate = 0.5
history = []
for i in range(40):
    g = grad_loss(D_estimate, C0_obs, C_observed, n_steps=500)
    D_estimate = D_estimate - learning_rate * g
    history.append(float(D_estimate))

print(f"True D:      {D_true}")
print(f"Recovered D: {D_estimate:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(history, label="D estimate")
plt.axhline(D_true, color="red", linestyle="--", label="True D")
plt.xlabel("Gradient descent step")
plt.ylabel("D")
plt.legend()
plt.title("Recovering the diffusion coefficient via jax.grad")
plt.show()
